# Task 3 — Web Crawler & Extractor

**Mata Kuliah:** Temu Kembali Informasi  
**Dataset:** Putusan Pidana Khusus Narkotika — Direktori Putusan MA RI  

Notebook ini melakukan akuisisi otomatis (crawling) berkas PDF putusan dari
`putusan3.mahkamahagung.go.id`, kemudian mengekstraksi metadata dan amar putusan
menggunakan RegEx ke dalam berkas `extracted_decisions.csv`.

## 0. Instalasi Dependensi

In [ ]:
!pip install -q requests==2.31.0 beautifulsoup4==4.12.3 pdfplumber==0.10.4 pandas==2.2.1

## 1. Ambil Repositori & Impor Modul

Ganti `Dataset-Narkotika_XXX_YYY` dengan nama repositori kelompok Anda.

In [ ]:
REPO_URL = "https://github.com/<username>/Dataset-Narkotika_XXX_YYY.git"
REPO_DIR = "Dataset-Narkotika_XXX_YYY"

import os, sys

if not os.path.isdir(REPO_DIR):
    !git clone $REPO_URL

sys.path.insert(0, os.path.abspath(REPO_DIR))
print("Working dir :", os.getcwd())
print("Repo ada    :", os.path.isdir(REPO_DIR))

## 2. Crawling Tautan Putusan

Tentukan kata kunci Pengadilan Negeri (mis. `Negeri Jakarta Barat`).

In [ ]:
from src.crawler import fetch_decision_links, fetch_pdf_url, download_pdf

PN_KEYWORD = "Negeri Jakarta Barat"
MAX_RESULTS = 50

links = fetch_decision_links(PN_KEYWORD, max_results=MAX_RESULTS)
print(f"Ditemukan {len(links)} tautan detail putusan.")
links[:5]

## 3. Unduh Berkas PDF

In [ ]:
import time
from src.crawler import _build_session

OUT_DIR = os.path.join(REPO_DIR, "Dataset")
os.makedirs(OUT_DIR, exist_ok=True)

session = _build_session()
downloaded = []
for i, detail_url in enumerate(links, 1):
    pdf_url = fetch_pdf_url(detail_url, session=session)
    if not pdf_url:
        print(f"[{i}] PDF tidak ditemukan")
        continue
    name = os.path.basename(pdf_url.split('?')[0]) or f'putusan_{i}.pdf'
    path = os.path.join(OUT_DIR, name)
    if download_pdf(pdf_url, path, session=session):
        downloaded.append(path)
    time.sleep(1)

print(f"\n{len(downloaded)} berkas PDF tersimpan di {OUT_DIR}")

## 4. Ekstraksi Metadata & Amar Putusan (RegEx)

In [ ]:
from src.extractor import extract_folder, save_csv

df = extract_folder(OUT_DIR)
csv_path = save_csv(df, os.path.join(REPO_DIR, "data", "extracted_decisions.csv"))

print("Tersimpan di:", csv_path)
df.head()

## 5. Pratinjau Hasil Ekstraksi

In [ ]:
print("Jumlah baris :", len(df))
print("Kolom        :", list(df.columns))
df[['File', 'No Putusan', 'Lembaga Peradilan']].head(10)